# DM4ML Assignment 2
## End-to-End Data Management Pipeline for a Recommendation System

This notebook presents actual outputs from the reusable Python pipeline modules: ingestion, validation, preparation, transformation, feature creation, model training, and evaluation. Saved outputs and trained models are reused when available.

## 1. Data Ingestion

The ingestion function downloads and standardizes MovieLens 100K user, movie, and rating tables. This cell reuses saved CSVs and the ingestion report, calling the function only if those outputs are missing.

In [1]:
import json
import os
from pathlib import Path
import sys

import pandas as pd

project_root_override = os.environ.get("DM4ML_PROJECT_ROOT")
root_candidates = []
if project_root_override:
    root_candidates.append(Path(project_root_override).expanduser())
root_candidates.extend([Path.cwd(), *Path.cwd().parents])
root_candidates.extend(Path(entry) for entry in sys.path if entry)
project_root = next(
    (candidate.resolve() for candidate in root_candidates
     if (candidate / "main.py").is_file() and (candidate / "src").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open the notebook from the project "
        "workspace or set DM4ML_PROJECT_ROOT to its directory."
    )
sys.path.insert(0, str(project_root))
from src.config import ProjectPaths
from src.ingestion.movielens import ingest_movielens

paths = ProjectPaths.from_root(project_root)
REFRESH_OUTPUTS = True
print(f"Refresh reports and derived outputs: {REFRESH_OUTPUTS}")
csv_paths = {name: paths.raw / f"{name}.csv" for name in ("users", "movies", "ratings")}
report_path = paths.reports / "ingestion_report.json"

if not all(path.exists() for path in csv_paths.values()) or not report_path.exists():
    ingestion_report = ingest_movielens(paths.raw, paths.reports)
else:
    ingestion_report = json.loads(report_path.read_text(encoding="utf-8"))

users = pd.read_csv(csv_paths["users"])
movies = pd.read_csv(csv_paths["movies"])
ratings = pd.read_csv(csv_paths["ratings"])
print(f"Ingestion status: {ingestion_report.get('status', 'not recorded')}")

Refresh reports and derived outputs: True
Ingestion status: success


In [2]:
ingestion_shapes = pd.DataFrame(
    {name: {"rows": len(frame), "columns": len(frame.columns)}
     for name, frame in {"users": users, "movies": movies, "ratings": ratings}.items()}
).T
display(ingestion_shapes)

display(users.head())
display(movies[["movie_id", "title", "release_date"]].head())
display(ratings.head())

,rows,columns
users,943,5
movies,1682,24
ratings,100000,4


,user_id,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


,movie_id,title,release_date
0,1,Toy Story (1995),01-Jan-1995
1,2,GoldenEye (1995),01-Jan-1995
2,3,Four Rooms (1995),01-Jan-1995
3,4,Get Shorty (1995),01-Jan-1995
4,5,Copycat (1995),01-Jan-1995


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
ingestion_summary = pd.DataFrame(
    [
        {"item": "status", "value": ingestion_report.get("status")},
        {"item": "source", "value": ingestion_report.get("source")},
        {"item": "users", "value": ingestion_report.get("record_counts", {}).get("users", len(users))},
        {"item": "movies", "value": ingestion_report.get("record_counts", {}).get("movies", len(movies))},
        {"item": "ratings", "value": ingestion_report.get("record_counts", {}).get("ratings", len(ratings))},
        {"item": "archive", "value": ingestion_report.get("archive")},
    ]
)
display(ingestion_summary)

,item,value
0,status,success
1,source,https://files.grouplens.org/datasets/movielens...
2,users,943
3,movies,1682
4,ratings,100000
5,archive,data/raw/ml-100k.zip


**Interpretation:** The tables and saved report show which source datasets were ingested and their record counts. Ingestion reads the saved CSVs on subsequent runs, avoiding another download or rewrite.

## 2. Data Validation

The validation module checks missing values, duplicates, types, rating bounds, timestamps, and referential integrity. It saves a validation report and a CSV of invalid records.

In [4]:
from src.validation import validate_data

validation_report_path = paths.reports / "validation_report.json"
rejected_records_path = paths.errors / "invalid_records.csv"
if not REFRESH_OUTPUTS and validation_report_path.exists() and rejected_records_path.exists():
    validation_report = json.loads(validation_report_path.read_text(encoding="utf-8"))
else:
    validation_report = validate_data(paths.raw, paths.reports, paths.errors)
display(pd.DataFrame(validation_report["checks"]))
display(pd.DataFrame([{
    "status": validation_report["status"],
    "warning_count": validation_report["warning_count"],
    "rejected_record_count": validation_report["rejected_record_count"],
    "invalid_records_file": str(rejected_records_path),
}]))
display(pd.DataFrame(validation_report["warnings"]))
invalid_records = pd.read_csv(rejected_records_path)
display(invalid_records.head())

Validation passed_with_warnings: 6 check(s), 0 rejected record(s)


,check,failure_count,passed
0,missing_values,0,True
1,duplicate_records,0,True
2,incorrect_data_types,0,True
3,rating_range_1_to_5,0,True
4,valid_unix_timestamps,0,True
5,user_movie_referential_integrity,0,True


,status,warning_count,rejected_record_count,invalid_records_file
0,passed_with_warnings,1686,0,/Users/tacky/Documents/BITS/Sem2/DM4ML/Assignm...


,table_name,field,issue,classification,record_count,source_rows,handling
0,movies,release_date,missing_optional_value,warning,1,[266],preserved and filled during preparation
1,movies,video_release_date,missing_optional_value,warning,1682,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",preserved and filled during preparation
2,movies,imdb_url,missing_optional_value,warning,3,"[266, 1357, 1358]",preserved and filled during preparation


,table_name,source_row,issues,user_id,age,gender,occupation,zip_code,movie_id,title,...,horror,musical,mystery,romance,sci_fi,thriller,war,western,rating,timestamp


**Interpretation:** Optional movie metadata gaps (including release dates) are reported as warnings and recovered during preparation; no records are rejected in this dataset. Rating range, timestamps, duplicate keys, data types, and referential integrity have zero failures.

## 3. Data Preparation

The preparation module cleans the ingested data and creates a chronological 80/20 rating split. All ratings before the time boundary are training data; later ratings are test data.

In [5]:
from src.preparation import prepare_data

preparation_report_path = paths.processed / "preparation_report.json"
if not REFRESH_OUTPUTS and preparation_report_path.exists():
    preparation_report = json.loads(preparation_report_path.read_text(encoding="utf-8"))
else:
    preparation_report = prepare_data(
        paths.raw, paths.processed,
        rejected_records_path=paths.errors / "invalid_records.csv",
    )
display(pd.DataFrame([preparation_report["clean_record_counts"]]))
display(pd.DataFrame([{
    "filled_missing_values": preparation_report["filled_missing_values"],
    "dropped_all_missing_movie_columns": preparation_report["dropped_all_missing_movie_columns"],
}]))
display(pd.DataFrame([preparation_report["split"]]))
ratings_train = pd.read_csv(paths.processed / "ratings_train.csv")
ratings_test = pd.read_csv(paths.processed / "ratings_test.csv")
display(ratings_train.head())
display(ratings_test.head())

,users,movies,ratings
0,943,1682,100000


,filled_missing_values,dropped_all_missing_movie_columns
0,"{'users.gender': 0, 'users.occupation': 0, 'us...",[video_release_date]


,method,requested_test_fraction,target_train_records,train_records,test_records,train_fraction,test_fraction,train_max_timestamp,test_min_timestamp,cutoff_timestamp,cutoff_group_assignment,cutoff_timestamp_in_train,cutoff_timestamp_in_test
0,chronological; all interactions with the cutof...,0.2,80000,79999,20001,0.79999,0.20001,889237247,889237269,889237269,test,False,True


,user_id,movie_id,rating,timestamp
0,259,255,4,874724710
1,259,286,4,874724727
2,259,298,4,874724754
3,259,185,4,874724781
4,259,173,4,874724843


,user_id,movie_id,rating,timestamp
0,3,322,3,889237269
1,3,323,2,889237269
2,3,335,1,889237269
3,3,349,3,889237269
4,3,264,2,889237297


**Interpretation:** The report shows cleaned row counts and the chronological boundary. All ratings with the cutoff timestamp are kept together, so the latest training timestamp is strictly earlier than the earliest test timestamp.

## 4. Data Transformation

The transformation module uses training ratings only to create user activity, movie popularity, and a user-movie interaction matrix.

In [6]:
from src.transformation import transform_data

transformation_report_path = paths.processed / "transformation_report.json"
if not REFRESH_OUTPUTS and transformation_report_path.exists():
    transformation_report = json.loads(transformation_report_path.read_text(encoding="utf-8"))
else:
    transformation_report = transform_data(paths.processed)
user_activity = pd.read_csv(paths.processed / "user_activity.csv")
movie_popularity = pd.read_csv(paths.processed / "movie_popularity.csv")
interaction_sample = pd.read_csv(paths.processed / "interaction_matrix.csv", index_col=0, nrows=5)
display(user_activity.head())
display(movie_popularity.sort_values("popularity_count", ascending=False).head())
display(interaction_sample.iloc[:, :8])
display(pd.DataFrame([transformation_report["shapes"]]))

,user_id,activity_count,user_average_rating,last_interaction_timestamp
0,1,265,3.611321,888732928
1,2,62,3.709677,888980240
2,3,31,2.322581,889237247
3,5,175,2.874286,879198898
4,6,211,3.635071,883717304


,movie_id,title,popularity_count,movie_average_rating
49,50,Star Wars (1977),473,4.355180
180,181,Return of the Jedi (1983),423,4.000000
99,100,Fargo (1996),417,4.153477
293,294,Liar Liar (1997),396,3.146465
257,258,Contact (1997),394,3.817259


,1,2,3,4,5,6,7,8
user_id,,,,,,,,
1,5.0,3.0,4.0,3.0,0.0,5.0,4.0,1.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


,user_activity,movie_popularity,interaction_matrix
0,"[751, 4]","[1682, 4]","[943, 1682]"


**Interpretation:** User and movie summaries and the interaction matrix are derived exclusively from training ratings. The matrix uses 0 to represent an unobserved training interaction.

## 5. Feature Management

The feature module creates reusable user, movie, and interaction features from the training split and writes a feature catalog documenting definitions, types, and sources.

In [7]:
from src.features import build_features

feature_report_path = paths.features / "feature_report.json"
if not REFRESH_OUTPUTS and feature_report_path.exists():
    feature_report = json.loads(feature_report_path.read_text(encoding="utf-8"))
else:
    feature_report = build_features(paths.processed, paths.features)
user_features = pd.read_csv(paths.features / "user_features.csv")
movie_features = pd.read_csv(paths.features / "movie_features.csv")
interaction_features = pd.read_csv(paths.features / "interaction_features.csv")
feature_catalog = pd.read_csv(paths.features / "feature_catalog.csv")
display(user_features.head())
display(movie_features[["movie_id", "title", "movie_popularity", "movie_average_rating"]].head())
display(interaction_features.head())
display(feature_catalog)

,user_id,age,gender,occupation,zip_code,user_average_rating,user_activity_count,last_interaction_timestamp,days_since_last_interaction,preferred_genre
0,1,24,M,technician,85711,3.611321,265,888732928.0,5.837025,drama
1,2,53,F,other,94043,3.709677,62,888980240.0,2.974618,drama
2,3,23,M,writer,32067,2.322581,31,889237247.0,0.000000,comedy
3,4,24,M,technician,43537,3.517656,0,NaN,-1.000000,unknown
4,5,33,F,other,15213,2.874286,175,879198898.0,116.184595,comedy


,movie_id,title,movie_popularity,movie_average_rating
0,1,Toy Story (1995),374,3.879679
1,2,GoldenEye (1995),110,3.218182
2,3,Four Rooms (1995),80,3.012500
3,4,Get Shorty (1995),175,3.514286
4,5,Copycat (1995),74,3.337838


,user_id,movie_id,rating,timestamp,days_since_previous_interaction
0,1,168,5,874965478,NaN
1,1,172,5,874965478,0.000000
2,1,165,5,874965518,0.000463
3,1,156,4,874965556,0.000440
4,1,166,5,874965677,0.001400


,feature_name,definition,data_type,source,entity
0,user_average_rating,Mean explicit training rating; global training...,float,ratings_train.csv,user
1,user_activity_count,Number of training ratings recorded for the user,integer,ratings_train.csv,user
2,preferred_genre,Most frequent genre among the user's training ...,string,ratings_train.csv + movies_clean.csv,user
3,days_since_last_interaction,Days between the user's last training rating a...,float,ratings_train.csv,user
4,movie_popularity,Number of training ratings for the movie,integer,ratings_train.csv,movie
5,movie_average_rating,Mean explicit training rating; global training...,float,ratings_train.csv,movie
6,days_since_previous_interaction,Elapsed days since the user's previous trainin...,float,ratings_train.csv,interaction


**Interpretation:** The user, movie, and interaction feature tables make common signals reusable, while the catalog records each feature's definition, type, entity, and source.

## 6. Model Training

The training module creates the popularity baseline and Surprise SVD model. It reuses saved model files when present and records parameters, metrics, and artifacts in local MLflow tracking.

In [8]:
from src.training import train_models

training_report_path = paths.reports / "model_training_report.json"
model_files = [paths.models / "popularity_model.pkl", paths.models / "svd_model.pkl"]
training_report = (
    json.loads(training_report_path.read_text(encoding="utf-8"))
    if training_report_path.exists() else {}
)
models_are_current = (
    all(path.exists() for path in model_files)
    and training_report.get("training_records") == len(ratings_train)
)
if not models_are_current:
    training_report = train_models(
        paths.processed, paths.models, paths.reports, paths.root / "mlruns"
    )
display(pd.DataFrame([
    {"model": item["model"], "run_id": item["run_id"], "artifact": item["artifact"], "parameters": item["parameters"]}
    for item in training_report["models"]
]))
display(pd.DataFrame([{
    "experiment": training_report["experiment_name"],
    "tracking_database": training_report["tracking_database"],
    "training_records": training_report["training_records"],
}]))

,model,run_id,artifact,parameters
0,popularity,942b1df4702d4983a2504fe378ce29e3,models/popularity_model.pkl,"{'algorithm': 'rating-count ranking', 'train_r..."
1,svd,cb5b5cc09e514a278a27f38a9603b8bc,models/svd_model.pkl,"{'algorithm': 'Surprise SVD', 'train_records':..."


,experiment,tracking_database,training_records
0,DM4ML-MovieLens100K,mlruns/mlflow.db,79999


**Interpretation:** The two saved models are trained on the training split only. Their parameters, MLflow run IDs, and serialized model artifact paths are shown above.

## 7. Model Evaluation

The evaluation module compares both saved models on the same held-out users and candidate items. It reports ranking metrics at K=10, catalog coverage, genre-based diversity, and explicit-rating RMSE/MAE.

In [9]:
from src.evaluation import evaluate_models

evaluation_report_path = paths.reports / "evaluation_report.json"
comparison_path = paths.reports / "evaluation_comparison.csv"
if not REFRESH_OUTPUTS and evaluation_report_path.exists() and comparison_path.exists():
    evaluation_report = json.loads(evaluation_report_path.read_text(encoding="utf-8"))
else:
    evaluation_report = evaluate_models(
        paths.processed, paths.models, paths.reports, paths.root / "mlruns", k=10
    )
evaluation_comparison = pd.read_csv(comparison_path)
display(evaluation_comparison)
display(pd.DataFrame([{
    "K": evaluation_report["k"],
    "relevant_rating_threshold": evaluation_report["relevant_rating_threshold"],
    "evaluation_users": evaluation_report["evaluation_user_count"],
    "coverage_denominator_movies": evaluation_report["coverage_denominator"],
}]))

,model,precision_at_10,recall_at_10,ndcg_at_10,coverage,diversity,rmse,mae
0,Popularity baseline,0.229310,0.084715,0.253700,0.045779,0.853398,1.036709,0.831423
1,SVD collaborative filtering,0.167931,0.057580,0.160557,0.073722,0.856333,1.028744,0.819445


,K,relevant_rating_threshold,evaluation_users,coverage_denominator_movies
0,10,4,290,1682


**Interpretation:** Higher Precision@10, Recall@10, NDCG@10, and Coverage indicate stronger ranking/reach; diversity measures genre spread, while lower RMSE/MAE indicate more accurate rating predictions. The users and test ratings follow the same relevance rule for both recommenders.

## Notebook status

All seven implemented stages are presented from their saved reports and datasets. The chronological test split is kept separate from transformations, features, and model fitting to avoid leakage.